# ND-AWS staircase — one dataset, one experiment set

Draws the iteration-by-iteration picture of a single NDAR run: the distribution of sampled
approximation ratios at each iteration, the mean and the best sample, and the staircase the
logged best traces. On a dataset with local-search refinement (HDLS), the refined samples are
drawn beside the quantum samples. The experiment set shown is the one with the highest
approximation ratio in the chosen dataset.

Two companion panels are produced from the same experiment set: the problem graph of the
Hamiltonian, and — optionally — the coupling map of the device.

Every sampled bitstring is re-evaluated against the rebuilt Hamiltonian and checked against
the logged summary, so a mismatch between the stored overview and the stored shots stops the
notebook.

**Writes**: `NDAWS_staircase[_WithBars].pdf/.svg`, `graph_example.pdf/.svg` and, when the
device panel is on, `<BACKEND_NAME>_graph.pdf/.svg`, all into
`./temp/staircases/<OUTPUT_SUBFOLDER>/` inside the notebook folder (ignored by git).
Nothing is written into the result store.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from quapopt.ancillary_functions.presets import get_standard_subfolders_hierarchy_full
from quapopt.circuits import backend_utilities as bck_utils
from quapopt.data_analysis.data_handling import (ResultsLogger,
                                                 STANDARD_NAMES_DATA_TYPES as SNDT,
                                                 STANDARD_NAMES_VARIABLES as SNV)
from quapopt.hamiltonians.generators import create_hamiltonian_from_descriptions
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

from quapopt.data_analysis.data_handling.io_utilities.dataset_management import gather_dataset_results
from quapopt.data_analysis.data_handling.io_utilities import DEFAULT_STORAGE_DIRECTORY

from quapopt.data_analysis.visualization.PlotlySubplotsPlotter import PlotlySubplotsPlotter

## Configuration

The only cell meant to be edited.

`DATASET_NAME` selects the dataset; `BACKEND_NAME` and `SIMULATION` must match the folder
the dataset was written under, and `HAM_CLASS_DESCRIPTION` must match its Hamiltonian class.
The defaults read the 100-qubit Erdos-Renyi QPU run.

`OUTPUT_SUBFOLDER` names the output folder under `./temp/staircases/`; it defaults to the
dataset name, so two datasets never overwrite each other's figures.

`PLOT_DEVICE_COUPLING_MAP` draws the coupling map of `BACKEND_NAME`. It fetches the device
live and therefore needs IBM credentials; it is off by default.

In [ ]:
# The naming rule of `00a_ndaws_main_run.ipynb`; set the RUN_ID your own study was written
# under. This default names the noise-directed arm of the 100-qubit ER-10 panel.
DATASET_NAME = 'NDAWS_QPU_k=0.25_ER-0.1_HDLS_run-01'
BACKEND_NAME = 'ibm_boston'
SIMULATION = False

HAM_CLASS_DESCRIPTION = "HMN=ER;LOC=(2,);CFD=CT~CON_CDN~NOR_CDP~loc~0.0_scale~1.0;ERT=GNP"
# 3-regular datasets: "HMN=RG;LOC=(2,);CFD=CT~CON_CDN~NOR_CDP~loc~0.0_scale~1.0"

OUTPUT_SUBFOLDER = DATASET_NAME

PLOT_DEVICE_COUPLING_MAP = False

In [ ]:
backend_name = BACKEND_NAME

simulation = SIMULATION
# The store keeps noiseless simulations apart from noisy ones, so the flag has to follow the
# dataset, not the SIMULATION switch: a damping run at q>0 is a simulation that is not
# noiseless, and reading it under a noiseless hierarchy finds nothing. The run notebooks put
# `noiseless` in the name of every dataset written without a noise model.
noiseless_simulation = ('noiseless' in DATASET_NAME) if simulation else False


ham_class_description = HAM_CLASS_DESCRIPTION
experiments_folders_hierarchy = get_standard_subfolders_hierarchy_full(experiment_category='NDAR',
                                                                       experiment_subcategory='WS-QAOA-p1',
                                                                       backend_name=backend_name,
                                                                       simulation=simulation,
                                                                       noiseless_simulation=noiseless_simulation,
                                                                       hamiltonian_class_description=ham_class_description
                                                                     )


dataset_name = DATASET_NAME
global_best_ar = 0.0
staircase_set_id = None
staircase_instance_description = None
for _df_overview, _metadata_experiment in gather_dataset_results(dataset_name=dataset_name,
                                                               data_types=[SNDT.NDAROverview, SNDT.CircuitsMetadata],
                                                               overwrite_storage_directory=DEFAULT_STORAGE_DIRECTORY,
                                                               overwrite_experiment_folders_hierarchy=experiments_folders_hierarchy,
                                                               return_none_if_not_found=True):

    if _df_overview is None or _metadata_experiment is None:
        continue

    _instance_description= _metadata_experiment['HamiltonianMetadata']['HamiltonianInstanceDescription']
    _hamiltonian = create_hamiltonian_from_descriptions(class_description=ham_class_description,
                                                            instance_description=_instance_description)

    _df_overview['ApproximationRatioBest'] = _hamiltonian.calculate_approximation_ratio(_df_overview['EnergyBest'].values)
    best_ar = _df_overview['ApproximationRatioBest'].max()


    if best_ar > global_best_ar:
        global_best_ar = best_ar
        staircase_set_id = _df_overview['ExperimentSetID'].values[0]
        staircase_instance_description = _instance_description
        print(f"New global best AR: {global_best_ar:.4f} in dataset {dataset_name}")




if staircase_set_id is None:
    raise ValueError(f"no readable experiment set found for dataset {dataset_name}; "
                     f"check the name, the backend {backend_name!r}, the simulation flag "
                     f"and the Hamiltonian class description")

print('best id:', staircase_set_id)
print('best instance:', staircase_instance_description)

In [ ]:
logger_kwargs_main = {'experiment_folders_hierarchy': experiments_folders_hierarchy,
                      'experiment_set_id': staircase_set_id,  #Used to group the experiments
                      }

results_reader = ResultsLogger(**logger_kwargs_main,
                                experiment_instance_id='nan')

metadata_experiment = results_reader.read_metadata(data_type=SNDT.CircuitsMetadata,
                                                    shared_across_experiment_set=True)
instance_description_staircase = metadata_experiment['HamiltonianMetadata']['HamiltonianInstanceDescription']

df_bitstrings = results_reader.gather_results(data_type=SNDT.Bitstrings,
                                             return_none_if_not_found=True,
                                             show_progress_bar=True,
                                             number_of_threads=1,
                                              # merge_instances_metadata_data_type=SNDT.
                                            )
df_overview = results_reader.gather_results(data_type=SNDT.NDAROverview,
                                             return_none_if_not_found=True,
                                             show_progress_bar=True,
                                             number_of_threads=1,
                                              # merge_instances_metadata_data_type=SNDT.
                                            )
df_bitstrings.sort_values(by=SNV.NDARIteration.id_long, ascending=True, inplace=True)
df_overview.sort_values(by=SNV.NDARIteration.id_long, ascending=True, inplace=True)

In [ ]:
hamiltonian_staircase_init = create_hamiltonian_from_descriptions(class_description=ham_class_description,
                                                            instance_description=instance_description_staircase)

number_of_qubits = hamiltonian_staircase_init.number_of_qubits

## The staircase

At each NDAR iteration the sampled bitstrings are re-scored with the Hamiltonian the sampler
used and drawn as a smoothed envelope; the mean (open circle) and the best sample (gold star)
are marked. Envelope colour follows the bias-parameter schedule, which the legend names.

An HDLS dataset (`HDLS` in its name) refined every shot by one local-search step (HDQS) and
logged the best refined sample, not the best shot. The notebook repeats that step on every
stored shot and draws both distributions: the quantum samples on the left half of each
envelope, the samples after HDQS on the right half in one fixed colour. Each half has its own
best-value star, gold for the quantum samples and violet after HDQS; a second legend above the
plot names them. Without HDLS the quantum samples fill the right half alone.

A dashed connector carries the logged best value, the one NDAR steps from, into the next
iteration. The assertions check that the stored per-shot data, the stored summary and the
re-computed energies agree.

In [ ]:
ndar_iterations = sorted(df_bitstrings['NDARIteration'].unique())

from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors
from quapopt.optimization.classical_solvers.LocalSearch.cython_implementation.local_swap_cython import \
    run_single_swap_choose_best_cython

plotly.io.templates.default = "plotly"
plotly.offline.init_notebook_mode(connected=True)

# The dataset name records how the run sampled, by the naming rule of the run notebooks. The
# Standard arm ('no-gauges-version') samples in the original frame with the original
# Hamiltonian; the noise-directed arm samples in the frame of the accumulated bitflip. An HDLS
# run refines every shot by one local-search step (HDQS) and logs the best refined sample as
# EnergyBest, while the stored shots stay unrefined.
standard_arm = 'no-gauges-version' in DATASET_NAME
add_hdls = 'HDLS' in DATASET_NAME

cost_hamiltonian_ndar = hamiltonian_staircase_init.copy()
cost_hamiltonian_ndar.reinitialize_backend(backend='numpy')
lowest_energy = cost_hamiltonian_ndar.lowest_energy
highest_energy = cost_hamiltonian_ndar.highest_energy
delta = highest_energy - lowest_energy
ground_state_bitstring = np.array(hamiltonian_staircase_init._lowest_energy_state)


bitflip_concatenated = np.array([0]*number_of_qubits)

staircase_figure_ar = make_subplots(specs=[[{"secondary_y": False}]])

staircase_figure_ar.update_layout(
   # title=f"{hamiltonian_staircase_init.hamiltonian_instance_description}; {backend_name} (QPU)",
    xaxis_title="Iteration",
    yaxis_title="Approximation Ratio",
    template="plotly",
    # font=dict(size=14),
    hovermode="x unified"
)



colors = dict(default="seagreen", first="indianred", last="royalblue")
colors2 = dict(default="seagreen", first="indianred", last="royalblue")
# The HDQS samples take one fixed colour; the quantum shots keep the bias-schedule colour.
hdqs_color = "mediumpurple"


# ---- cartoonish envelope styling knobs ----
envelope_kde_bw       = 0.2     # KDE bandwidth factor (raise -> smoother / rounder)
envelope_max_width    = 0.8     # max horizontal extent per iteration
envelope_fill_alpha   = 0.45
envelope_outline_w    = 3.5
envelope_n_points     = 400     # resolution of the envelope curve along y
envelope_bottom_q     = 0.005   # clip envelope bottom at this AR quantile (top stays at max)
shadow_offset_x       = 0.00
shadow_offset_y       = 0.00
shadow_alpha          = 0.25
# -------------------------------------------

add_histogram_bars = True


def add_half_envelope(figure, ars, x_spine, side, color):
    """Draw the envelope of the approximation ratios `ars` on one side of the spine at x_spine
    (side=1 to the right, side=-1 to the left): histogram bars when add_histogram_bars is on, a
    drop shadow, the filled envelope and its outline."""
    r_, g_, b_, _a_ = mcolors.to_rgba(color)
    fillcolor = f'rgba({int(255*r_)},{int(255*g_)},{int(255*b_)},{envelope_fill_alpha})'

    # ---- smoothed KDE envelope, clipped to observed AR range ----
    # bottom clipped at a quantile to avoid being dragged down by low-AR outliers
    ar_min = float(np.quantile(ars, envelope_bottom_q))
    ar_max = float(np.max(ars))
    y_grid = np.linspace(ar_min, ar_max, envelope_n_points)

    try:
        kde = gaussian_kde(ars, bw_method=envelope_kde_bw)
        density = kde(y_grid)
    except (np.linalg.LinAlgError, ValueError):
        # all-identical fallback: narrow gaussian bump at the mean
        mu = float(np.mean(ars))
        density = np.exp(-((y_grid - mu) ** 2) / (2 * 0.005 ** 2))

    density = density / np.max(density)
    widths  = side * density * envelope_max_width

    # closed polygon (spine on one side, curve on the other) -- used for fill + shadow
    x_poly = np.concatenate([[x_spine], x_spine + widths, [x_spine]])
    y_poly = np.concatenate([[y_grid[0]], y_grid, [y_grid[-1]]])


    if add_histogram_bars:
        # A sample at the optimum can score a ratio a rounding step above 1.0, outside the last
        # bin; clipping keeps it in the top bar.
        counts, edges = np.histogram(np.clip(ars, 0.0, 1.0),
                                     bins=100,
                                     range=(0.0, 1.0))
        counts = counts.astype(float)
        highest_count = np.max(counts)
        counts*=1/highest_count
        ycenters = 0.5*(edges[:-1] + edges[1:])

        for c, yc in zip(counts, ycenters):
            if c == 0:
                continue
            x0, x1 = x_spine, x_spine + side*c*0.8
            figure.add_trace(go.Scatter(
                x=[x0, x1],
                y=[yc, yc],
                mode="lines",
                line=dict(width=4, color=fillcolor),
                showlegend=False,
                hoverinfo="skip"
            ))




    # 1) drop shadow underneath (fill only, no outline -> no visible spine)
    figure.add_trace(go.Scatter(
        x=x_poly + shadow_offset_x,
        y=y_poly + shadow_offset_y,
        fill='toself',
        fillcolor=f'rgba(0,0,0,{shadow_alpha})',
        line=dict(width=0),
        mode='lines',
        showlegend=False,
        hoverinfo='skip',
    ))

    # 2) main filled envelope (fill only, no outline)
    figure.add_trace(go.Scatter(
        x=x_poly,
        y=y_poly,
        fill='toself',
        fillcolor=fillcolor,
        line=dict(width=0),
        mode='lines',
        showlegend=False,
        hoverinfo='skip',
    ))

    # 3) outline along the envelope curve only (no spine -> no vertical line)
    figure.add_trace(go.Scatter(
        x=x_spine + widths,
        y=y_grid,
        mode='lines',
        line=dict(width=envelope_outline_w, color=color),
        showlegend=False,
        hoverinfo='skip',
    ))


for ndar_iteration in ndar_iterations:
    df_i = df_bitstrings[df_bitstrings['NDARIteration']==ndar_iteration].copy()
    df_i_overview = df_overview[df_overview['NDARIteration']==ndar_iteration].copy()

    energy_best_i = df_i_overview['EnergyBest'].values[0]
    bitflip_best_i = df_i_overview['Bitflip'].values[0]

    if isinstance(bitflip_best_i,str):
        bitflip_best_i = eval(bitflip_best_i)

    all_bitstrings_physical_i = np.array(df_i['Bitstring'].tolist())
    # The Hamiltonian the sampler used, and the shots in the original frame (logical) and in the
    # frame the logged bitflips live in.
    if standard_arm:
        sampler_hamiltonian_i = hamiltonian_staircase_init
        all_bitstrings_logical_i = all_bitstrings_physical_i
        all_bitstrings_ndar_frame_i = all_bitstrings_physical_i^bitflip_concatenated
    else:
        sampler_hamiltonian_i = cost_hamiltonian_ndar
        all_bitstrings_logical_i = all_bitstrings_physical_i^bitflip_concatenated
        all_bitstrings_ndar_frame_i = all_bitstrings_physical_i
    all_energies_i = sampler_hamiltonian_i.evaluate_energy(all_bitstrings_physical_i)
    all_ars_i = cost_hamiltonian_ndar.calculate_approximation_ratio(all_energies_i)
    idx_sort = np.argsort(all_energies_i)

    energies_sorted_i = all_energies_i[idx_sort]
    bitstrings_sorted_i = all_bitstrings_ndar_frame_i[idx_sort]

    if add_hdls:
        # The sampler's HDQS step repeated on every stored shot, with the Hamiltonian the sampler
        # used, before this iteration's bitflip is applied.
        csr_indptr, csr_indices, csr_data = sampler_hamiltonian_i.get_csr_arrays()
        hdqs_bitstrings_pm_i = np.array([run_single_swap_choose_best_cython(bitstring_pm=bitstring_pm,
                                                                            csr_indptr=csr_indptr,
                                                                            csr_indices=csr_indices,
                                                                            csr_data=csr_data,
                                                                            tolerance_1swap=0.0,
                                                                            tolerance_2swap=0.0)
                                         for bitstring_pm in 1 - 2*all_bitstrings_physical_i.astype(np.int32)])
        hdqs_energies_i = sampler_hamiltonian_i.evaluate_energy(bitstrings_array=hdqs_bitstrings_pm_i,
                                                                pm_input=True)
        hdqs_ars_i = cost_hamiltonian_ndar.calculate_approximation_ratio(hdqs_energies_i)

    if not standard_arm:
        cost_hamiltonian_ndar.apply_bitflip(bitflip_best_i)


    bitflip_concatenated = bitflip_concatenated^np.array(bitflip_best_i)
    test_energy = hamiltonian_staircase_init.evaluate_energy([bitflip_concatenated])[0]

    all_energies_i_test = hamiltonian_staircase_init.evaluate_energy(all_bitstrings_logical_i)

    #test consistency -- shouldn't be necessary
    # With HDQS the logged best is the best refined sample, which can beat every stored shot; the
    # best stored shot is the logged bitflip only when the two energies are equal.
    assert energy_best_i <= energies_sorted_i[0] + 10**(-6), "Logged best energy above the best stored shot"
    if abs(energies_sorted_i[0] - energy_best_i)<10**(-6):
        assert tuple(bitstrings_sorted_i[0]) == tuple(bitflip_best_i), "Mismatch in bitflip best"
    if add_hdls:
        assert abs(np.min(hdqs_energies_i) - energy_best_i)<10**(-6), "Mismatch in HDQS energy best"
    assert abs(test_energy - energy_best_i)<10**(-6), "Mismatch in energy best"
    assert np.allclose(all_energies_i_test, all_energies_i), "Mismatch in logical/physuical energies"



    if ndar_iteration == 0:
        color = colors["first"]
        color2 = colors2["first"]
    elif ndar_iteration in [1,2,3]:
        color = colors["default"]
        color2 = colors2["default"]
    else:
        color = colors["last"]
        color2 = colors2["last"]

    number_of_shots = len(all_ars_i)


    #now let's make two lines. One will be vertical from y0 = mean_ar to y1 = best_ar
    mean_ar_i = np.mean(all_ars_i)
    best_ar_i = np.max(all_ars_i)
    if add_hdls:
        best_ar_hdqs_i = np.max(hdqs_ars_i)

    if ndar_iteration== min(ndar_iterations):
        worst_ar = np.min(all_ars_i)

    connecting_line_width = 7.5
    connecting_line_color = 'grey'
    connecting_line_dash = 'dash'

    # The mean is a small open circle and the best sample a larger gold star, both outlined
    # in black: the two markers sit on the same envelope and have to read apart at a glance.
    mean_marker_size = 22
    mean_marker_color = 'white'
    mean_marker_line_color = 'black'
    mean_marker_line_width = 3

    best_marker_size = 40
    best_marker_color = 'gold'
    best_marker_line_color = 'black'
    best_marker_line_width = 2

    # With HDQS each half carries its own markers, set off from the spine so that equal values
    # do not hide each other.
    hdqs_best_marker_color = 'darkviolet'
    half_marker_offset = 0.2

    # Without HDQS the shots fill the right half as the only distribution; with HDQS they fill
    # the left half and the HDQS samples the right.
    add_half_envelope(staircase_figure_ar, all_ars_i, ndar_iteration, -1 if add_hdls else 1, color)
    if add_hdls:
        add_half_envelope(staircase_figure_ar, hdqs_ars_i, ndar_iteration, 1, hdqs_color)

    print('iteration:', ndar_iteration, 'mean_ar:', mean_ar_i, 'best_ar:', best_ar_i)
    if add_hdls:
        print('iteration:', ndar_iteration, 'best_ar after HDQS:', best_ar_hdqs_i)

    # 4) staircase connector (drawn above envelope so it stays visible). NDAR steps from the
    # logged best, which with HDQS is the best refined sample.
    staircase_ar_i = best_ar_hdqs_i if add_hdls else best_ar_i
    if ndar_iteration!=max(ndar_iterations):
        #plot horizontal line from x0 = ndar_iteration to x1 = ndar_iteration + 0.8 at height staircase_ar_i
        y0 = staircase_ar_i
        y1 = staircase_ar_i
        x0, x1 = ndar_iteration, ndar_iteration + 1
        staircase_figure_ar.add_trace(go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode="lines",

            line=dict(width=connecting_line_width, color=connecting_line_color, dash=connecting_line_dash),
            showlegend=False,
        hoverinfo="skip",
        ))
    if ndar_iteration == min(ndar_iterations):
        # layer='below' so the reference line sits *under* the data markers/traces
        staircase_figure_ar.add_hline(y=1.0, line_dash="dash", line_color="black", line_width=7.5, layer="below")


    # 5) mean & best markers last -> drawn on top of envelope
    x_markers_shots = ndar_iteration - half_marker_offset if add_hdls else ndar_iteration
    staircase_figure_ar.add_trace(go.Scatter(
        x=[x_markers_shots],
        y=[mean_ar_i],
        mode="markers",
        marker=dict(size=mean_marker_size,
                    color=mean_marker_color,
                    line=dict(width=mean_marker_line_width, color=mean_marker_line_color)),
        name=f"Mean AR; It {ndar_iteration}",
        showlegend=False

    ))
    staircase_figure_ar.add_trace(go.Scatter(
        x=[x_markers_shots],
        y=[best_ar_i],
        mode="markers",
        marker=dict(size=best_marker_size,
                    color=best_marker_color,
                    symbol='star',
                    line=dict(width=best_marker_line_width, color=best_marker_line_color)),
        name=f"Best AR; It {ndar_iteration}",
        showlegend=False
    ))
    if add_hdls:
        staircase_figure_ar.add_trace(go.Scatter(
            x=[ndar_iteration + half_marker_offset],
            y=[best_ar_hdqs_i],
            mode="markers",
            marker=dict(size=best_marker_size,
                        color=hdqs_best_marker_color,
                        symbol='star',
                        line=dict(width=best_marker_line_width, color=best_marker_line_color)),
            name=f"Best AR after HDQS; It {ndar_iteration}",
            showlegend=False
        ))




# What the three colours encode is the bias-parameter schedule, not the iteration number, so
# the legend names the schedule. Iteration 0 runs unbiased at c=0.5; from then on each
# iteration implements two bias values and keeps the better result. Change this alongside the
# schedule in the run notebook.
bias_schedule_legend = [('first', 'c = 0.5'),
                        ('default', 'c ∈ {0.1, 0.05}'),
                        ('last', 'c ∈ {0.05, 0.025}')]

for group_key, group_label in bias_schedule_legend:
    r_, g_, b_, _a_ = mcolors.to_rgba(colors[group_key])
    staircase_figure_ar.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=40,
                    symbol='square',
                    color=f'rgba({int(255*r_)},{int(255*g_)},{int(255*b_)},{envelope_fill_alpha})',
                    line=dict(width=3, color=colors[group_key])),
        name=group_label,
        showlegend=True,
        hoverinfo='skip',
    ))

# With HDQS a second legend, above the plot, names the two halves and their stars.
if add_hdls:
    r_, g_, b_, _a_ = mcolors.to_rgba(hdqs_color)
    halves_legend = [(dict(symbol='star', color=best_marker_color), 'best quantum sample'),
                     (dict(symbol='square', color=f'rgba({int(255*r_)},{int(255*g_)},{int(255*b_)},{envelope_fill_alpha})'),
                      'after HDQS'),
                     (dict(symbol='star', color=hdqs_best_marker_color), 'best after HDQS')]
    for marker_style, label in halves_legend:
        staircase_figure_ar.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            marker=dict(size=40,
                        line=dict(width=3, color=hdqs_color if label == 'after HDQS' else best_marker_line_color),
                        **marker_style),
            name=label,
            legend='legend2',
            showlegend=True,
            hoverinfo='skip',
        ))


#let's update xticks to only NDAR iterations:
staircase_figure_ar.update_xaxes(tickvals=ndar_iterations,
                                 range=[-1, max(ndar_iterations)+1])
staircase_figure_ar.update_yaxes(tickvals=np.linspace(0, 1.0, 11),
                                 range=[worst_ar*0.98, 1.02],
                                 secondary_y=False
                                 )
staircase_figure_ar.update_yaxes(#tickvals=np.linspace(0, 1.0, 11),
                                 range=[-1, number_of_qubits//2+2],
        secondary_y=True
)


staircase_figure_ar.update_yaxes(range=(0.45, 1.05))

staircase_figure_ar = PlotlySubplotsPlotter.update_figure_with_standard_specs(figure=staircase_figure_ar)

staircase_figure_ar.update_layout(height=int(1500 * 0.7),
                     width=int(3000 * 0.7))

# Set after the standard specs so neither the legend box nor its font is overwritten. The
# box sits bottom right, the one corner the rising staircase leaves free.
staircase_figure_ar.update_layout(
    showlegend=True,
    legend=dict(title=dict(text='bias parameter'),
                x=0.99,
                y=0.02,
                xanchor='right',
                yanchor='bottom',
                bgcolor='white',
                bordercolor='black',
                borderwidth=2,
                itemsizing='constant',
                font=dict(size=58, family='Serif'),
                title_font=dict(size=58, family='Serif')),
)
if add_hdls:
    staircase_figure_ar.update_layout(
        legend2=dict(orientation='h',
                     x=0.0,
                     y=1.0,
                     xanchor='left',
                     yanchor='bottom',
                     bgcolor='white',
                     itemsizing='constant',
                     font=dict(size=58, family='Serif')),
    )


main_exp_info = OUTPUT_SUBFOLDER

folder_path = f'./temp/staircases/{main_exp_info}/'
os.makedirs(folder_path, exist_ok=True)

figure_name = f"NDAWS_staircase"

if add_histogram_bars:
    figure_name+='_WithBars'

file_path_full = f"{folder_path}/{figure_name}"
extensions_list = ['pdf', 'svg']
for extension in extensions_list:
    if extension in ['png', 'pdf', 'svg']:
        plotly.io.write_image(staircase_figure_ar, f'{file_path_full}.{extension}', format=extension, scale=2.)

    elif extension == 'html':
        plotly.offline.plot(staircase_figure_ar,filename= f'{file_path_full}.{extension}')

    else:
        raise ValueError(f"Invalid extension: {extension}")

## Problem graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

folder_path = f'./temp/staircases/{main_exp_info}/'
os.makedirs(folder_path, exist_ok=True)

figure_name = 'graph_example'
file_path_full = f"{folder_path}/{figure_name}"


graph_nx = nx.Graph()

for (i,j) in hamiltonian_staircase_init.get_edges(False):
    graph_nx.add_edge(i,j)


ax = plt.gca() # to get the current axis
# fig, ax = plt.subplots(1, 1, figsize=(20, 20))
nx.draw(graph_nx,
        #pos=perf_graph_cords,
        #labels=labels_linear if plot_linear_labels else None,
        with_labels=True,
        node_color="lightblue",
        node_size=750,
        ax=ax,
        linewidths=2.0,
        width=3.0,
        edge_color='grey'
        )

ax.collections[0].set_edgecolor("black")

#set size of figure
plt.gcf().set_size_inches(20, 20)


extensions_list = ['pdf', 'svg']

for extension in extensions_list:
    if extension in ['png', 'pdf', 'svg']:
        plt.savefig(f'{file_path_full}.{extension}', format=extension, dpi=300)
    else:
        raise ValueError(f"Invalid extension: {extension}")

## Device coupling map

Off by default: it fetches the device live and needs IBM credentials.

In [ ]:
if PLOT_DEVICE_COUPLING_MAP:
    from qiskit.transpiler.coupling import CouplingMap
    provider_ibm = bck_utils.get_qiskit_provider()
    backend = provider_ibm.backend(backend_name)
    coupling_map:CouplingMap = backend.coupling_map   # CouplingMap object
    edges = coupling_map.get_edges()      # list of (control, target) tuples
    n = backend.num_qubits

In [ ]:
if PLOT_DEVICE_COUPLING_MAP:
    from collections import defaultdict, deque
    def grid_positions_from_coupling(edges):
        """Integer (x,y) per qubit, from coupling map alone.
        (i, i+1) edge => horizontal; anything else => vertical.
        Coordinates propagated by BFS from the lowest-index qubit."""
        adj = defaultdict(list)
        for u, v in edges:
            a, b = min(u, v), max(u, v)
            kind = "H" if b == a + 1 else "V"
            adj[a].append((b, kind))
            adj[b].append((a, kind))

        start = min(adj)
        pos = {start: (0, 0)}
        dq = deque([start])
        while dq:
            cur = dq.popleft()
            cx, cy = pos[cur]
            for nb, kind in adj[cur]:
                if nb in pos:
                    continue
                if kind == "H":
                    pos[nb] = (cx + 1 if nb > cur else cx - 1, cy)
                else:
                    pos[nb] = (cx, cy - 1 if nb > cur else cy + 1)
                dq.append(nb)
        return pos



    graph_ibm_nx = nx.Graph()
    edges_ibm = list(coupling_map.get_edges())
    graph_ibm_nx.add_edges_from(edges_ibm)
    pos = grid_positions_from_coupling(edges_ibm)


    ax = plt.gca() # to get the current axis
    # fig, ax = plt.subplots(1, 1, figsize=(20, 20))
    nx.draw(graph_ibm_nx,
            #pos=perf_graph_cords,
            #labels=labels_linear if plot_linear_labels else None,
            with_labels=True,
            node_color="lightblue",
            node_size=750,
            ax=ax,
            linewidths=2.0,
            width=10.0,
            edge_color='grey',
            pos=pos
            )

    ax.collections[0].set_edgecolor("black")

    #set size of figure
    plt.gcf().set_size_inches(15, 15)


    # nx.draw(graph_ibm_nx, pos=pos, with_labels=True, node_color="lightblue", node_size=750)
    folder_path = f'./temp/staircases/{main_exp_info}/'
    os.makedirs(folder_path, exist_ok=True)

    figure_name = f'{backend_name}_graph'
    file_path_full = f"{folder_path}/{figure_name}"


    extensions_list = ['pdf', 'svg']

    for extension in extensions_list:
        if extension in ['png', 'pdf', 'svg']:
            plt.savefig(f'{file_path_full}.{extension}', format=extension, dpi=300)
        else:
            raise ValueError(f"Invalid extension: {extension}")